<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Validation_on_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

In [2]:
SEED = 42

import random

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
df = pd.read_excel("HPVVAL.xlsx")

print(df.shape)

df.head()

(726, 17)


,PatientID,CenterID,Task 1,Task 2,Task 3,Age,Gender,Tobacco Consumption,Alcohol Consumption,Performance Status,Treatment,T-stage,N-stage,M-stage,HPV Status,Relapse,RFS
0,CHUM-001,1,1,1,0,82.0,1,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1704.0
1,CHUM-002,1,1,1,0,73.0,1,NaN,NaN,NaN,1.0,T3,N2,M0,NaN,1.0,439.0
2,CHUM-006,1,1,1,0,65.0,1,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1186.0
3,CHUM-007,1,1,1,0,70.0,0,NaN,NaN,NaN,0.0,T2,N2,M0,NaN,0.0,1702.0
4,CHUM-008,1,1,1,0,67.0,0,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1499.0


In [4]:
df = df.dropna(subset=['HPV Status'])

In [5]:
tobacco_mode = df['Tobacco Consumption'].mode()[0]

df['Tobacco Consumption'] = df[
    'Tobacco Consumption'
].fillna(tobacco_mode)

In [6]:
alcohol_mode = df['Alcohol Consumption'].mode()[0]

df['Alcohol Consumption'] = df[
    'Alcohol Consumption'
].fillna(alcohol_mode)

In [7]:
df = df.drop(
    columns=[
        'Performance Status',
        'Relapse',
        'RFS',
        'PatientID',
        'CenterID',
        'Task 1',
        'Task 2',
        'Task 3'
    ]
)

In [8]:
df = df.dropna()

print(df.shape)

(562, 9)


In [9]:
df['T-stage'] = df['T-stage'].replace({
    'T0':0,
    'T1':1,
    'T2':2,
    'T3':3,
    'T4':4
})

df['N-stage'] = df['N-stage'].replace({
    'N0':0,
    'N1':1,
    'N2':2,
    'N3':3
})

df['M-stage'] = df['M-stage'].replace({
    'M0':0,
    'M1':1
})

/tmp/ipykernel_17589/963169859.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['T-stage'] = df['T-stage'].replace({
/tmp/ipykernel_17589/963169859.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['N-stage'] = df['N-stage'].replace({
/tmp/ipykernel_17589/963169859.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_sile

In [10]:
X = df[
    [
        'Age',
        'Gender',
        'Tobacco Consumption',
        'Alcohol Consumption',
        'Treatment',
        'T-stage',
        'N-stage',
        'M-stage'
    ]
]

y = df['HPV Status']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED
)

In [12]:
print(y_train.value_counts())

print()

print(y_test.value_counts())

HPV Status
1.0    407
0.0     42
Name: count, dtype: int64

HPV Status
1.0    99
0.0    14
Name: count, dtype: int64


In [13]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [14]:
smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [15]:
print(
    pd.Series(y_train_smote).value_counts()
)

HPV Status
1.0    407
0.0    407
Name: count, dtype: int64


In [16]:
X_train_smote = torch.FloatTensor(
    X_train_smote
)

y_train_smote = torch.LongTensor(
    y_train_smote.to_numpy()
)

X_test = torch.FloatTensor(
    X_test
)

y_test = torch.LongTensor(
    y_test.to_numpy()
)

In [17]:
class HPVNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(8,64)
        self.fc2 = nn.Linear(64,32)
        self.fc3 = nn.Linear(32,16)
        self.fc4 = nn.Linear(16,2)

        self.relu = nn.ReLU()

    def forward(self,x):

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))

        x = self.fc4(x)

        return x

In [18]:
model = HPVNet()

weights = torch.tensor(
    [3.0,1.0],
    dtype=torch.float32
)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [19]:
epochs = 500

best_val_loss = float('inf')

best_epoch = 0

for epoch in range(epochs):

    model.train()

    outputs = model(X_train_smote)

    train_loss = criterion(
        outputs,
        y_train_smote
    )

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        test_outputs = model(X_test)

        test_loss = criterion(
            test_outputs,
            y_test
        )

    if test_loss.item() < best_val_loss:

        best_val_loss = test_loss.item()

        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

    if (epoch+1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}"
        )

print("Best Test Loss =", best_val_loss)

print("Best Epoch =", best_epoch)

Epoch 50, Train=0.4334, Test=0.6857
Epoch 100, Train=0.2494, Test=0.7716
Epoch 150, Train=0.1937, Test=0.9307
Epoch 200, Train=0.1474, Test=1.2923
Epoch 250, Train=0.1093, Test=2.0048
Epoch 300, Train=0.0856, Test=2.7815
Epoch 350, Train=0.0706, Test=3.5593
Epoch 400, Train=0.0621, Test=4.3589
Epoch 450, Train=0.0563, Test=5.0470
Epoch 500, Train=0.0518, Test=5.7203
Best Test Loss = 0.6438533067703247
Best Epoch = 66


In [20]:
model.load_state_dict(
    torch.load("best_model.pth")
)

<All keys matched successfully>

In [21]:
model.eval()

with torch.no_grad():

    outputs = model(X_test)

    predicted = torch.argmax(
        outputs,
        dim=1
    )

In [22]:
print(
    classification_report(
        y_test.numpy(),
        predicted.numpy()
    )
)

              precision    recall  f1-score   support

           0       0.22      0.71      0.34        14
           1       0.94      0.65      0.77        99

    accuracy                           0.65       113
   macro avg       0.58      0.68      0.55       113
weighted avg       0.85      0.65      0.71       113



In [23]:
cm = confusion_matrix(
    y_test.numpy(),
    predicted.numpy()
)

print(cm)

[[10  4]
 [35 64]]


In [24]:
with torch.no_grad():

    outputs = model(X_test)

    probs = torch.softmax(
        outputs,
        dim=1
    )

    hpv_positive_probs = probs[:,1]

auc = roc_auc_score(
    y_test.numpy(),
    hpv_positive_probs.numpy()
)

print("AUC =", auc)

AUC = 0.7626262626262627


In [25]:
print("Best Test Loss =", best_val_loss)

print("Best Epoch =", best_epoch)

Best Test Loss = 0.6438533067703247
Best Epoch = 66
